# Chapter 05b — SVD, Low-Rank Approximation & PCA

Eigenvalues only exist for square matrices. The **Singular Value Decomposition
(SVD)** generalizes the idea to *any* matrix and is arguably the most useful
factorization in all of applied mathematics:

$$A = U\,\Sigma\,V^{\top}.$$

- $U$ and $V$ have **orthonormal columns** (rotations/reflections),
- $\Sigma$ is diagonal with non-negative **singular values**
  $\sigma_1 \ge \sigma_2 \ge \dots \ge 0$ (pure stretches).

Every linear map is therefore *rotate → stretch → rotate*. In this notebook we
use the SVD for **low-rank approximation** (compression) and then build
**PCA** from scratch.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

## 2. Computing the SVD with `np.linalg.svd`

By default NumPy returns the "full" factors. We usually want the compact form,
so pass `full_matrices=False`: then for an $m\times n$ matrix the shapes are
$U:(m,k)$, $S:(k,)$, $V^{\top}:(k,n)$ with $k=\min(m,n)$.

Note that NumPy returns $V^{\top}$ (already transposed), not $V$.

In [ ]:
A = np.array([[3.0, 1.0, 1.0],
              [1.0, 3.0, 1.0]])

U, S, Vt = np.linalg.svd(A, full_matrices=False)
print("U shape :", U.shape)
print("S (singular values):", S)   # always >= 0, descending
print("Vt shape:", Vt.shape)

Reconstruct $A$ from the pieces. Since $S$ is returned as a 1-D array, we turn it
into a diagonal matrix with `np.diag` before multiplying.

In [ ]:
A_rebuilt = U @ np.diag(S) @ Vt
print("reconstruction matches A?", np.allclose(A_rebuilt, A))   # True

## 3. Low-rank approximation: keep the biggest singular values

The SVD writes $A$ as a sum of rank-1 layers, each weighted by a singular value:

$$A = \sum_{i=1}^{k} \sigma_i\, u_i\, v_i^{\top}.$$

Because the $\sigma_i$ are sorted largest-first, the **top few** terms already
capture most of the matrix. Keeping only the first $r$ terms gives the *best*
rank-$r$ approximation (this is the Eckart–Young theorem).

Let's build a small grayscale "image" with structure and watch the approximation
sharpen as we add singular values.

In [ ]:
# A 20x20 "image": a smooth ramp plus a couple of bright stripes.
n = 20
x = np.linspace(0, 1, n)
img = np.outer(x, x)              # smooth gradient (rank 1 on its own)
img[5, :] += 0.8                  # a bright horizontal stripe
img[:, 12] += 0.6                 # a bright vertical stripe

U, S, Vt = np.linalg.svd(img, full_matrices=False)
print("number of singular values:", len(S))
print("first few singular values:", S[:6])

In [ ]:
def low_rank(U, S, Vt, r):
    """Reconstruct using only the top r singular values."""
    return U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :]

ranks = [1, 2, 4, len(S)]
fig, axes = plt.subplots(1, len(ranks), figsize=(12, 3.2))
for ax, r in zip(axes, ranks):
    approx = low_rank(U, S, Vt, r)
    ax.imshow(approx, cmap='gray')
    ax.set_title(f"rank {r}")
    ax.axis('off')
plt.tight_layout()
plt.show()

The rank-1 picture is blurry; by rank 4 it is already very close to the full
image. Let's quantify the shrinking error.

In [ ]:
for r in range(1, len(S) + 1):
    approx = low_rank(U, S, Vt, r)
    err = np.linalg.norm(img - approx)     # Frobenius (overall) error
    if r <= 6 or r == len(S):
        print(f"rank {r:2d}:  reconstruction error = {err:.4f}")

The error drops fast and reaches (essentially) zero once $r$ equals the true
rank. That is compression: store a few singular values and vectors instead of
the whole matrix.

## 4. PCA from scratch — the data

**Principal Component Analysis** finds the directions along which a cloud of
points varies the most. We make a 2-D cloud that is clearly stretched along a
slanted direction.

In [ ]:
# 200 points: independent spread, then stretched and rotated into a tilt.
N = 200
base = rng.standard_normal((N, 2)) * np.array([2.0, 0.5])   # wide, then thin
theta = np.deg2rad(35)                                       # rotate 35 degrees
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
data = base @ R.T + np.array([5.0, 3.0])                     # rotate + shift

plt.figure(figsize=(5, 5))
plt.scatter(data[:, 0], data[:, 1], s=12, alpha=0.6)
plt.gca().set_aspect('equal'); plt.grid(True)
plt.title("Raw 2-D point cloud")
plt.show()

## 5. Step 1 — center the data

PCA is about *spread around the mean*, so we first subtract the mean of each
column. The centered cloud sits at the origin.

In [ ]:
mean = data.mean(axis=0)            # average of each column (x and y)
X = data - mean                     # centered data
print("original mean:", mean)
print("centered mean:", X.mean(axis=0))   # ~ [0, 0]

## 6. Step 2 — covariance matrix and its eigenvectors

The **covariance matrix** $C = \frac{1}{N-1} X^{\top} X$ summarizes how the
coordinates vary together. Its **eigenvectors** are the principal axes and its
**eigenvalues** are the variance along each axis. Because $C$ is symmetric, we
use `np.linalg.eigh`.

In [ ]:
C = (X.T @ X) / (len(X) - 1)        # 2x2 covariance matrix
print("covariance matrix:")
print(C)

vals, vecs = np.linalg.eigh(C)      # ascending order
# Reorder so the LARGEST variance comes first.
order = np.argsort(vals)[::-1]
vals = vals[order]
vecs = vecs[:, order]
print("variances (eigenvalues):", vals)
print("principal axes (columns):")
print(vecs)

The same axes come straight out of the SVD of the centered data — no covariance
matrix needed. This is why PCA and SVD are two views of one idea.

In [ ]:
U, S, Vt = np.linalg.svd(X, full_matrices=False)
# Rows of Vt are the principal axes; variances are S^2 / (N-1).
print("variances from SVD :", S**2 / (len(X) - 1))
print("axes from SVD (rows of Vt):")
print(Vt)

## 7. Step 3 — draw the principal axes over the data

Each principal axis is drawn from the mean, scaled by the standard deviation
($\sqrt{\text{variance}}$) so its length reflects how much the data spreads that
way.

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(data[:, 0], data[:, 1], s=12, alpha=0.5)

colors = ['tab:red', 'tab:green']
for i in range(2):
    axis = vecs[:, i] * np.sqrt(vals[i]) * 2.5   # scale for visibility
    plt.quiver(mean[0], mean[1], axis[0], axis[1],
               angles='xy', scale_units='xy', scale=1,
               color=colors[i], width=0.012,
               label=f"PC{i+1} (var={vals[i]:.2f})")

plt.gca().set_aspect('equal'); plt.grid(True)
plt.legend(loc='upper left')
plt.title("Principal axes over the data")
plt.show()

The longer red arrow (PC1) lies along the direction of greatest spread; the
shorter green arrow (PC2) is perpendicular to it.

## 8. Step 4 — project to 1-D and report variance explained

Projecting the centered data onto PC1 reduces it from 2 numbers to 1 while
keeping as much spread as possible. The **fraction of variance explained** by a
component is its eigenvalue divided by the total.

In [ ]:
pc1 = vecs[:, 0]                 # top principal direction
proj1d = X @ pc1                 # one number per point: its position along PC1

total_var = vals.sum()
explained = vals / total_var
print(f"PC1 explains {explained[0]*100:.1f}% of the variance")
print(f"PC2 explains {explained[1]*100:.1f}% of the variance")

In [ ]:
# Visualize the 1-D projection laid out on a line.
plt.figure(figsize=(7, 1.8))
plt.scatter(proj1d, np.zeros_like(proj1d), s=12, alpha=0.5)
plt.yticks([])
plt.xlabel("position along PC1")
plt.title(f"Data reduced to 1-D  ({explained[0]*100:.1f}% variance kept)")
plt.grid(True, axis='x')
plt.show()

Most of the spread survives in a single dimension — that is dimensionality
reduction in action, and it is exactly how PCA compresses high-dimensional data
in machine learning.

---
## ✍️ Exercise 1 (solution included)

The squared **Frobenius norm** of a matrix equals the sum of the squares of its
singular values: $\|A\|_F^2 = \sum_i \sigma_i^2$. Verify this for a random
$5\times 4$ matrix.

**Solution:**

In [ ]:
A = rng.standard_normal((5, 4))
S = np.linalg.svd(A, compute_uv=False)   # singular values only

lhs = np.linalg.norm(A)**2               # Frobenius norm, squared
rhs = np.sum(S**2)
print("||A||_F^2       :", lhs)
print("sum of sigma^2  :", rhs)
print("match?", np.allclose(lhs, rhs))

## ✍️ Exercise 2 (solution included)

Make a fresh 2-D cloud stretched mostly along one direction. Use the SVD route
(not the covariance matrix) to compute the principal axes, and report what
percentage of the variance the **first** component explains.

**Solution:**

In [ ]:
# Strongly anisotropic cloud: big spread in x, tiny in y, then rotate.
cloud = rng.standard_normal((150, 2)) * np.array([3.0, 0.4])
ang = np.deg2rad(20)
Rot = np.array([[np.cos(ang), -np.sin(ang)],
                [np.sin(ang),  np.cos(ang)]])
cloud = cloud @ Rot.T

Xc = cloud - cloud.mean(axis=0)          # center
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
variances = S**2 / (len(Xc) - 1)
explained = variances / variances.sum()

print("variances :", variances)
print(f"PC1 explains {explained[0]*100:.1f}% of the variance")

---
## 📝 Homework (no solutions provided)

1. For a random $6\times 6$ matrix, compute its SVD and plot the singular values
   `S` with `plt.plot`. How quickly do they decay?
2. Build a low-rank matrix on purpose: `A = np.outer(u, v) + np.outer(p, q)`
   for random vectors. Confirm via the SVD that it has rank 2 (only two
   singular values are non-zero, up to tiny numerical noise).
3. Take the 20×20 "image" from Section 3 and find the smallest rank $r$ whose
   reconstruction error is below `0.1`. How much storage does that save versus
   the full matrix?
4. Generate a 3-D point cloud that lies (noisily) near a single plane. Use PCA
   to find the two dominant directions and report the combined variance
   explained by the top two principal components.